# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/surajprai2111-ui/ML-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### My ranked content-action queue

I use the transparent baseline scoring approach developed in ML-07 to rank eligible content pages for human review. The score combines four components: search visibility, freshness risk, position opportunity, and content depth.

The four components have the following weights:

- Search visibility: 40%
- Freshness risk: 30%
- Position opportunity: 25%
- Content depth: 5%

The resulting baseline action score is used to rank pages from highest to lowest priority. A higher score means that a page deserves earlier inspection under these rules; it does not prove that the page is declining or that an update will improve its performance.

I also assign reason codes to explain which conditions triggered a review recommendation. These include meaningful search visibility, a stale but visible page, an observed declining trend with visibility, a position opportunity, low click-through rate despite visibility, and thin content with visibility.

The action is assigned using a rule-based mapping of these reason codes:

- `investigate_decline`: investigate an observed declining trend on a page with visibility.
- `review_title_meta`: review the title and metadata when a visible page has a low click-through rate.
- `review_content_freshness`: review whether an older, visible page needs updating.
- `review_search_intent`: investigate whether the page matches the search intent.
- `assess_content_depth`: assess whether a relatively short page provides sufficient information.
- `general_review`: conduct an initial review when none of the specific action rules applies.

When several conditions apply, the action-mapping rules determine which action is assigned first. The reason codes retain information about the conditions that triggered the review.

The queue contains 30,000 eligible pages with 30,000 unique content IDs. The top 20 pages are displayed as a sample of the highest-ranked items.

The ranking and action labels are intended to support human review. They are not proof that a page needs changing, and they do not establish that a recommended intervention will improve traffic or business outcomes.

In [10]:
import pandas as pd
import numpy as np

# 1. Load the starter dataset
url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

print("Original dataset shape:", df.shape)


# 2. Keep eligible content pages
base = df.loc[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].drop_duplicates(subset="content_id").copy()


# 3. Convert the score-related columns to numeric values
numeric_columns = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "days_since_last_update",
    "content_age_days",
]

for col in numeric_columns:
    base[col] = pd.to_numeric(base[col], errors="coerce")

print("Eligible pages:", len(base))
print("Unique content IDs:", base["content_id"].nunique())


# 4. Calculate the four transparent score components

# Visibility: higher impressions receive a higher score.
base["visibility_score"] = (
    np.log1p(base["impressions_90d"])
    / np.log1p(base["impressions_90d"].max())
) * 100


# Freshness risk: older updates receive a higher score.
# Missing update age is not assumed to mean stale.
base["freshness_risk_score"] = (
    base["days_since_last_update"].clip(lower=0, upper=365) / 365
) * 100

base["freshness_risk_score"] = (
    base["freshness_risk_score"].fillna(0)
)


# Position opportunity: pages in positions 1–20 receive a score.
position = base["avg_position"]

base["position_opportunity_score"] = np.where(
    position.between(1, 20),
    ((position - 1) / 19) * 100,
    0,
)


# Content depth: shorter pages receive a higher review score.
base["depth_gap_score"] = np.select(
    [
        base["word_count"].between(1, 599),
        base["word_count"].between(600, 1199),
    ],
    [
        100,
        50,
    ],
    default=0,
)


# 5. Calculate the baseline action score using the ML-07 weights
base["baseline_action_score"] = (
    0.40 * base["visibility_score"]
    + 0.30 * base["freshness_risk_score"]
    + 0.25 * base["position_opportunity_score"]
    + 0.05 * base["depth_gap_score"]
).round(2)


# 6. Assign reason codes to explain why each page deserves review
reason_codes = []

for _, row in base.iterrows():
    reasons = []

    if row["impressions_90d"] >= 500:
        reasons.append("meaningful_visibility")

    if (
        pd.notna(row["days_since_last_update"])
        and row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        row["trend_direction"] == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_visibility")

    if (
        pd.notna(row["avg_position"])
        and 0 < row["avg_position"] <= 20
    ):
        reasons.append("position_opportunity")

    if (
        row["impressions_90d"] >= 500
        and pd.notna(row["avg_position"])
        and 0 < row["avg_position"] <= 20
        and pd.notna(row["ctr"])
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        pd.notna(row["word_count"])
        and 0 < row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if not reasons:
        reasons.append("general_review")

    reason_codes.append("|".join(reasons))

base["reason_codes"] = reason_codes


# 7. Assign a suggested action based on the reason codes.
# These are recommendations for human review, not automatic changes.

def assign_action(reason_codes):
    reasons = set(reason_codes.split("|"))

    if "declining_with_visibility" in reasons:
        return "investigate_decline"

    elif "low_ctr_visible_page" in reasons:
        return "review_title_meta"

    elif "stale_visible_page" in reasons:
        return "review_content_freshness"

    elif "position_opportunity" in reasons:
        return "review_search_intent"

    elif "thin_visible_page" in reasons:
        return "assess_content_depth"

    else:
        return "general_review"


base["action"] = base["reason_codes"].apply(assign_action)


# 8. Rank pages from highest to lowest score
output = base.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

output.insert(0, "rank", range(1, len(output) + 1))


# Keep a convenient Top-20 queue for inspection and later export.
top20 = output.head(20).copy()


# 9. Display the results
display_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "action",
    "reason_codes",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

display(top20[display_columns])


# 10. Validate the ranked queue
print("\nRanked queue rows:", len(output))
print("Top-20 rows:", len(top20))
print(
    "Duplicate content IDs in queue:",
    output["content_id"].duplicated().sum()
)
print(
    "Actions assigned:",
    output["action"].value_counts().to_dict()
)

print(
    "Missing action values:",
    output["action"].isna().sum()
)

print(
    "Missing reason codes:",
    output["reason_codes"].isna().sum()
)

Original dataset shape: (30000, 44)
Eligible pages: 30000
Unique content IDs: 30000


,rank,content_id,baseline_action_score,action,reason_codes,impressions_90d,avg_position,ctr,word_count
0,1,content_cf56e2e2e282,74.08,investigate_decline,meaningful_visibility|stale_visible_page|decli...,61678,19.7,0.15,5125.0
1,2,content_6a5b8ccbd700,67.52,review_search_intent,meaningful_visibility|position_opportunity,148534,18.3,0.60,5853.0
2,3,content_b9f7afeded79,67.22,investigate_decline,meaningful_visibility|declining_with_visibilit...,95333,19.1,0.31,6929.0
3,4,content_29c0b99ed170,65.81,review_title_meta,meaningful_visibility|position_opportunity|low...,40577,20.0,0.41,6660.0
4,5,content_eb366e871254,65.65,review_title_meta,meaningful_visibility|position_opportunity|low...,168060,16.6,0.20,5527.0
5,6,content_c2d929d83eaa,65.25,investigate_decline,meaningful_visibility|stale_visible_page|decli...,7558,17.9,0.20,4758.0
6,7,content_147ae2b19152,65.19,review_title_meta,meaningful_visibility|position_opportunity|low...,37755,19.7,0.30,6517.0
7,8,content_65fde7375946,64.93,review_title_meta,meaningful_visibility|position_opportunity|low...,53398,18.7,0.21,5093.0
8,9,content_4b05c87e3f08,64.84,review_title_meta,meaningful_visibility|position_opportunity|low...,29516,20.0,0.47,5094.0
9,10,content_003e0d52bd3d,64.75,review_title_meta,meaningful_visibility|position_opportunity|low...,37122,19.4,0.18,5960.0



Ranked queue rows: 30000
Top-20 rows: 20
Duplicate content IDs in queue: 0
Actions assigned: {'investigate_decline': 13152, 'review_search_intent': 7270, 'general_review': 5918, 'review_title_meta': 3639, 'assess_content_depth': 20, 'review_content_freshness': 1}
Missing action values: 0
Missing reason codes: 0


In [11]:
# Verify all action categories and their counts

action_counts = output["action"].value_counts(dropna=False)

display(action_counts.rename_axis("action").reset_index(name="count"))

print("Total action counts:", action_counts.sum())
print("Total queue rows:", len(output))
print("Action counts match queue rows:", action_counts.sum() == len(output))

print("\nReason-code examples:")
display(output[["content_id", "action", "reason_codes"]].head(10))

,action,count
0,investigate_decline,13152
1,review_search_intent,7270
2,general_review,5918
3,review_title_meta,3639
4,assess_content_depth,20
5,review_content_freshness,1


Total action counts: 30000
Total queue rows: 30000
Action counts match queue rows: True

Reason-code examples:


,content_id,action,reason_codes
0,content_cf56e2e2e282,investigate_decline,meaningful_visibility|stale_visible_page|decli...
1,content_6a5b8ccbd700,review_search_intent,meaningful_visibility|position_opportunity
2,content_b9f7afeded79,investigate_decline,meaningful_visibility|declining_with_visibilit...
3,content_29c0b99ed170,review_title_meta,meaningful_visibility|position_opportunity|low...
4,content_eb366e871254,review_title_meta,meaningful_visibility|position_opportunity|low...
5,content_c2d929d83eaa,investigate_decline,meaningful_visibility|stale_visible_page|decli...
6,content_147ae2b19152,review_title_meta,meaningful_visibility|position_opportunity|low...
7,content_65fde7375946,review_title_meta,meaningful_visibility|position_opportunity|low...
8,content_4b05c87e3f08,review_title_meta,meaningful_visibility|position_opportunity|low...
9,content_003e0d52bd3d,review_title_meta,meaningful_visibility|position_opportunity|low...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Who should use this playbook?

This playbook is intended for content analysts and SEO teams who need to prioritise pages for human review. It converts baseline scoring rules into a ranked queue, helping reviewers decide which pages deserve attention first.

### Intended use

The queue ranks eligible content pages using the baseline action score and provides reason codes and suggested actions to explain why a page was flagged.

The current output contains 30,000 eligible pages and 30,000 unique content IDs. The top 20 pages are displayed as a sample of the highest-ranked items.

The current action counts are:

- `investigate_decline`: 13,152 pages
- `review_search_intent`: 7,270 pages
- `general_review`: 5,918 pages
- `review_title_meta`: 3,639 pages
- `assess_content_depth`: 20 pages
- `review_content_freshness`: 1 page

These counts sum to 30,000 pages. They describe the output of the current rule-based action mapping, not the number of pages that have been manually reviewed or confirmed to need a particular intervention.

The queue should therefore be treated as a starting point for review prioritisation, not as an automatic decision about which pages must be changed.

### What the playbook can support

- Prioritising pages for an initial human review.
- Showing reason codes that help reviewers understand which rules triggered a recommendation.
- Helping content teams organise their review workload.
- Supporting decisions alongside business context, content quality, search intent, and editorial judgement.

### Cost and value considerations

Reviewing and updating content requires time and resources, so the highest-ranked page should not automatically receive the most expensive intervention.

The team should consider the potential business value of a page, its current performance, the likely effort required for review, and the relevance of the content to the intended audience.

For example:

- A page with substantial impressions and a possible ranking opportunity may deserve an initial review because it already receives search visibility.
- A page with declining visibility may need investigation before the team decides whether to update it.
- A page requiring substantial rewriting should be assessed against its business importance and the estimated effort before resources are committed.

The current playbook does not calculate financial return, expected traffic gains, or the cost of individual actions. These factors have not been measured in this notebook, so the ranking should be treated as a prioritisation aid rather than a proven return-on-investment estimate.

A human reviewer should weigh potential value against effort and confirm the appropriate action before work begins.

### Limits and assumptions

- The baseline score is a prioritisation heuristic, not proof that a page will improve after being refreshed.
- A high score does not guarantee that a page is declining or that updating it will increase traffic.
- The action labels are assigned using fixed rules. They are not predictions from a trained machine-learning model.
- The action counts depend on the order and conditions of the current action-mapping rules. They should not be interpreted as independently verified counts of pages that genuinely need each intervention.
- The queue uses the available dataset and its measured features. It does not establish that the same ranking will perform equally well for every client or in future periods.
- Reason codes explain which rules triggered a flag; they do not establish that the flagged condition caused poor performance.
- The queue should be reviewed against content age, search intent, business value, editorial quality, and other relevant context before action is taken.

### Decision rule

Use the ranking to decide **which pages to inspect first**, not which pages to change automatically. A human reviewer should confirm the reason for each recommendation and choose the final action based on the page's context.

The playbook is a research and decision-support prototype. It is not a production system, and its recommendations should not be applied automatically.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review and the no-go list

Before acting on a recommendation, a reviewer should::

- **Verify the reason codes:** Check which rules triggered the recommendation and confirm that the underlying measurements are meaningful and sufficiently complete.
- **Review search intent:** Confirm that the page matches what users are looking for and serves the intended audience.
- **Check content quality:** Review accuracy, usefulness, originality, readability, and whether the information is up to date.
- **Consider business value:** Check the page's importance to the business, its role in conversions, and its relevance to strategic goals before deciding what to change.
- **Review content age and context:** A recently published page may not have had enough time to demonstrate its performance. An older page is not necessarily outdated.
- **Check the action mapping:** Confirm that the suggested action makes sense given all the reason codes and the page's circumstances.
- **Choose the appropriate action:** Decide whether the page needs refreshing, further investigation, monitoring, or no change.

### What should never be automated

The playbook must not automatically:

- Delete or unpublish a page because it receives a high score.
- Rewrite or refresh content without human review.
- Change important factual, medical, financial, legal, or other sensitive information without qualified review.
- Remove content solely because its traffic, clicks, or impressions are low.
- Treat a reason code or an observed declining trend as proof that a particular issue caused poor performance.
- Assume that updating a page will increase traffic or improve business outcomes.
- Make changes without considering search intent, content quality, business value, and the page's wider context.

### Human decision rule

The ranked queue should help reviewers decide **which pages to inspect first**, not automatically decide which pages to change.

The current output contains 30,000 eligible pages. The action labels are generated by fixed rules, and the reason codes explain which conditions triggered those rules. Neither the score nor the suggested action independently confirms that a page needs a particular intervention.

Reviewers should use the ranking and reason codes as supporting information, check the underlying evidence, and make the final decision based on the context of each page.

This playbook is a decision-support prototype, not an autonomous content-editing system. Human approval is required before any content change is made.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Why monitoring is necessary

Content performance, search behaviour, competition, and business priorities can change over time. A ranking that was useful when it was created may become less useful as these conditions change.

The playbook should therefore be reviewed periodically. Monitoring should help identify when the existing scoring rules, action mapping, or recommendations need investigation.

### Signals that recommendations may have become stale

1. **Changes in content performance:** Impressions, clicks, CTR, or average position change substantially compared with a previous, comparable monitoring period.

2. **Changes in the dataset:** The distribution of important features, such as impressions, content age, CTR, and average position, differs noticeably from the data used to create the ranking.

3. **Changes in content or search behaviour:** Search intent, competition, or the relevance of existing content changes enough to affect the usefulness of previous recommendations.

4. **Poor reviewer agreement:** Content reviewers frequently disagree with the suggested priorities or find that the reason codes do not match the page's actual needs.

5. **Weak outcomes after review:** Pages selected for review do not show the expected improvement after changes are made. This should trigger investigation. It does not, by itself, prove that the scoring rules failed or that the content changes caused the observed outcome.

6. **Changes in business priorities:** A page that was previously important may become less valuable, or a new business objective may change which pages deserve attention first.

7. **Changes in ranking or action distribution:** The share of pages receiving particular actions, or the composition of the highest-ranked pages, changes substantially between monitoring periods.

### What action should follow?

- **Investigate first:** Check whether the change is real, whether the data is complete, and whether the monitoring period is comparable with earlier periods.
- **Review the rules:** If the baseline scoring rules no longer reflect useful priorities, assess and revise them.
- **Re-evaluate the ranking:** Compare the updated queue with reviewer feedback, the reason codes, and relevant content outcomes.
- **Review the action mapping:** Check whether the rules still produce useful and meaningful action categories. A change in action counts alone does not establish that the recommendations have become better or worse.
- **Retrain only when applicable:** The current playbook uses a baseline scoring rule rather than a trained machine-learning model. Retraining is therefore not currently applicable. If a trained model is introduced later, retraining should be considered when monitoring shows meaningful deterioration and sufficient new labelled data is available.
- **Validate before reuse:** Test any revised rules or future model on suitable data before relying on the updated recommendations.

### Monitoring limitations

The current notebook does not establish numerical thresholds for data drift, ranking quality, reviewer agreement, or business outcomes. The signals above are proposed monitoring checks, not measured evidence that drift or deterioration has occurred.

The notebook also does not establish that changes made after a recommendation caused any subsequent improvement or decline in performance.

The playbook should not automatically change content or retrain a future model when a single metric changes. A human should investigate the evidence and approve any material change to the scoring rules or model.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
from pathlib import Path

# Create the output folder if it does not exist
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the complete ranked content queue
queue_path = output_dir / "content_action_queue.csv"
output.to_csv(queue_path, index=False)

# Verify the exported file
print("Queue exported successfully:", queue_path)
print("Rows exported:", len(output))
print("Unique content IDs:", output["content_id"].nunique())
print("Duplicate content IDs:", output["content_id"].duplicated().sum())
print("Actions assigned:", output["action"].value_counts().to_dict())

Queue exported successfully: work/outputs/content_action_queue.csv
Rows exported: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
Actions assigned: {'investigate_decline': 13152, 'review_search_intent': 7270, 'general_review': 5918, 'review_title_meta': 3639, 'assess_content_depth': 20, 'review_content_freshness': 1}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.